# 03C1 - Encoder-decoder worked details

The [main chapter](03C-recurrent-encoder-decoder-to-attention.ipynb) keeps the core conceptual explanations and just three
small implementation cells. This independent notebook preserves the complete original
`ReaderWriter` experiment: initialized-state probes, teacher forcing, gradient interventions,
the encoder-only proof, matched summary/attention training, held-out evidence and trainer adapter.
The familiar source and separate reversal objective do not change.

**Coming from the main chapter?** Treat this as an experimental workbench,
not another required primer. The retained recap below lets it run independently;
you can use the jump links to go straight to the question you want to test.

| New question | Experiment |
|---|---|
| Does changing only the previous answer alter the next scores? | [Previous-input intervention](#previous-input) |
| Does the encoder really receive feedback? | [Connected/detached gradients](#gradients) |
| Can the note improve while the decoder stays fixed? | [Encoder-only update](#encoder-only) |
| What do supplied versus generated histories hide? | [Held-out checking](#evaluation) |
| Does source access help this same task? | [Attention comparison](#attention) and [routing](#routing) |

**Same task, separate implementation.** Main 03C uses a sequence GRU and prepared
previous-input tensors. This `ReaderWriter` uses one-step `GRUCell` writer calls
and constructs its practice prefix inside `forward`. Its benchmark generates a
fixed source-length-plus-one horizon; the main generation helper stops on `<eos>`
or its safety budget. Do not mix the two state interfaces or treat their runs as
one model continued in place.

**Run All from a fresh kernel.** No checkpoint or download is required. The original seeded
split algorithm below is retained and checked against the exact frozen corpus used by the
main demo. Both original comparison models still receive 1,000 updates with the same batches.
Their 76.43% / 97.50% results belong to this experiment, not the main `EncoderDecoder` demo.
Running this notebook refreshes only the original measured 03C figures and evidence.

**Jump to:** [roles](#roles), [teacher forcing](#teacher-forcing), [gradients](#gradients),
[encoder-only proof](#encoder-only), [attention](#attention),
[evaluation](#evaluation), [trainer adapter](#adapter).
**Code-only view:** [03C1-encoder-decoder-worked-details.py](03C1-encoder-decoder-worked-details.py).



## Optional recap - How do a reader and writer learn to work together?

This retained recap supports standalone reading. If you have just finished main
03C, the experiment menu above is your route to the additional evidence.

[03B](03B-counts-to-recurrent-language-models.ipynb) let one running state continue a word stream.
Now keep **the cat sat on the mat**, but ask for a separate answer:
**mat the on sat cat the**, then `<eos>`.
Reversing words is a deliberately checkable reading/writing drill, not natural English or translation.

**Why this drill?** We could reverse a list with one line of Python. Instead,
we ask the reader and writer to learn the job so we can inspect how they pass information.
The writer needs the last input word first, then the others in the right order.
We know the exact answer, so we can test the handoff, the teaching signals,
and the benefit of looking back without arguing about whether an answer sounds good.

You should leave able to follow what the reader produces, what starts the writer, which
previous word the writer receives, and how an answer's error teaches both components.
Only after that will we ask whether the writer should be allowed to reopen source positions.

**Code-only view:** [Python companion](03C1-encoder-decoder-worked-details.py).
**Notes:** [Handwritten companion](03C-recurrent-encoder-decoder-to-attention-theory.md).


## The questions we need to answer

An **encoder** does the reading job; a **decoder** does the writing job.
Those names tell you their roles, not which kind of neural layer they must use.
Here both use recurrent machinery you already understand from the memory chapter.

| Question | What we will inspect |
|---|---|
| Why isn't the decoder just another encoder? | Different inputs, responsibilities and learned weight sets |
| What crosses their boundary? | A computed state vector, not a word or copied weights |
| What does teacher forcing actually force? | The next input, not the current guess |
| How does the reader learn without predicting an answer itself? | The writer's error travels back across the handoff |
| Does backward already train the weights? | Compare backward alone with one optimizer step |
| What does attention change? | Which reader states the writer may consult at each step |

All state is rebuilt here. No checkpoint or download is needed.
We first inspect one small initialized model, then train the original two bounded comparisons.
The initialized probe demonstrates wiring and feedback, not language skill.


## 1. Reading and writing are different jobs

Suppose you must reverse the cat sentence. You cannot write its first answer, `mat`,
until you know how the source ends. So first **read the source**, then **write the answer**.

| Part | What arrives | What it produces |
|---|---|---|
| Shared word table | Word IDs | Trainable word features |
| Encoder / reader | Source features in source order | Contextual notes, including a final summary |
| Decoder / writer | Source context and a previous answer input | An updated writing state |
| Prediction head | The writing state | Scores for possible next answer words |

The encoder processes `the`, `cat`, `sat`, `on`, `the`, `mat`. It does not select an answer word
after every source word in this task. Its useful output is context for the writer.
The decoder starts a **different sequence** and must choose `mat`, then `the`, and so on.

They can use the same GRU update idea while learning **different weights**:
the reader learns what source information to retain; the writer learns how to use it while writing.
One word table is shared here, but the reader GRU, writer GRU and prediction head have their own parameters.

### Who tells the encoder what a good encoding looks like?

**Nobody supplies a correct encoding. We supply the correct answer.**
Think of the encoder writing a note for the decoder. At first, neither has learned
a useful way to communicate. We grade the answer they produce together, not the note itself.
A useful note is one the decoder can use to do the requested job.

In the one-card model, consider these two lessons:

| Source read by encoder | First decoder input | Required first answer |
|---|---|---|
| the cat sat on the mat | `<bos>` | mat |
| the cat sat on the rug | `<bos>` | rug |

If the encoder sends **exactly the same note** for both, the decoder gets the same
note and the same `<bos>` input. It must produce the same scores for two different
required answers. No clever decoder can recover a distinction the note has removed.
Different notes are necessary here, though merely being different is not enough.

Training makes that complaint actionable:

1. Read the source and compute a note with the current encoder weights.
2. Let the decoder use it and score an answer. Suppose it gives `mat` too little probability.
3. Backpropagation asks which small changes to the note would lower that answer's error,
   **through the decoder as it currently works**. It does not hand over a target note.
4. Continue that feedback through the source reads to the encoder weights that produced the note.
5. Update the weights and read again. The encoder now computes a changed note; in ordinary
   training, the decoder's weights also improve how it uses notes.

So the encoder and decoder learn a way to communicate **together**. We do not decide that
coordinate 7 must mean 'last word'. Many numerical representations could work.
For reversal, the useful distinctions include word identity and order; another task
could reward different details. The encoding is useful **for its task and decoder**,
not an automatically perfect description of the input.
Section 4.5 isolates this: keep the decoder fixed and let only the encoder improve its note.

**Read the picture as a handoff, not a word lookup:** the reader's final computed note crosses
to the writer. That note is not the literal word `mat`, the source sentence, or a copied set of weights.

![The reader processes all six source words, then its final learned summary initializes a separate writer.](images/03-evo-04-one-card-bottleneck.png)

Both states have 64 numbers in this model, so the reader's final state can become the writer's
initial state. Matching widths makes that handoff possible; it does not make their jobs or weights identical.
After writing starts, the writer changes its own state. It does not run the source reader again at every word.


<a id="teacher-forcing"></a>

## Worked details: Teacher forcing and answer alignment


## 2. Teacher forcing keeps the training example on track

When a decoder writes word by word, each new prediction depends on the earlier words
supplied to it. During generation, those words are its own guesses. Early in training,
the guesses are unreliable. Feed a wrong guess into the next step, and one mistake
can send the rest of the answer off track.

Think of a student misreading step 1 of a maths problem, then being marked wrong on
steps 2 through 7 while following that mistaken start. We want to teach each step of
the intended solution, not only practise the consequences of the first mistake.

**Teacher forcing separates grading from choosing the next input:**

1. **Grade the model's actual prediction.** Given `<bos>` and source context, the first
   target is `mat`. If the highest-scoring guess is `cat`, grade those scores against
   `mat` normally. Do not rewrite the prediction or erase its error.
2. **Feed the true previous word into the next step.** Instead of passing the guessed
   `cat` forward, supply `mat`, the previous word from the correct answer.
   This keeps the next practice question on its intended path.

The teacher is the known training answer, not another model whispering the next word.
The writer keeps its computed state; supplying `mat` does not reset that state or update weights.

| Step | Input fed to decoder | Target word graded |
|---|---|---|
| 1 | `<bos>` | `mat` |
| 2 | `mat` | `the` |
| 3 | `the` | `on` |
| 4 | `on` | `sat` |
| 5 | `sat` | `cat` |
| 6 | `cat` | `the` |
| 7 | `the` | `<eos>` |

Notice two things:

- **No cheating on the current word.** The target is one step ahead of the input.
  The decoder's answer inputs contain only earlier target words, never the current
  target or later ones. The encoder may read the entire known source, including `mat`;
  source context is allowed and is different from leaking the current answer input.
- **Target length drives practice.** Reversal happens to need six words plus `<eos>`.
  Translation or summarization can produce a shorter or longer answer than the source.
  The supplied target sequence determines how many writing steps we train.

Every position gets its intended previous-word input, so it remains a useful practice
question even when an earlier prediction was wrong. This chapter's GRU still computes
states sequentially: teacher forcing changes the supplied words, not that dependency.


**Before the picture:** use the same source in both panels.
For this controlled illustration, suppose the first guess is `cat` rather than `mat`.
Only the previous-word input policy changes; this is not a measured model prediction.

![Practice uses the correct previous answer; generation feeds back its own guess. Neither policy repairs the earlier wrong scores.](images/03-evo-06-practice-vs-generation.png)

**Trace the second input with your finger:** practice supplies `mat`; generation supplies `cat`.
Both keep the already-computed writing state. Neither empties the source context.
The different input can change the next state and scores; it need not ruin every later word.


In [ ]:
import json
import random
import time
from pathlib import Path
from unittest.mock import patch
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from language_data import CAT_SENTENCE, CAT_WORDS, CAT_FAMILY_WORDS
random.seed(17); np.random.seed(17); torch.manual_seed(17)
torch.set_num_threads(1)
started = time.perf_counter()
PAD = 0
word_to_id = {word: index + 1 for index, word in enumerate(CAT_FAMILY_WORDS)}
BOS, EOS = len(word_to_id) + 1, len(word_to_id) + 2
V = len(word_to_id) + 3
id_to_word = {index: word for word, index in word_to_id.items()}
id_to_word.update({PAD: "<pad>", BOS: "<bos>", EOS: "<eos>"})


def encode_words(words):
    return [word_to_id[word] for word in words]


def decode_ids(ids):
    return [id_to_word[int(index)] for index in ids]


def reversal_targets(source):
    return torch.cat([source.flip(1), source.new_full((len(source), 1), EOS)], dim=1)


def practice_prefix(target):
    return torch.cat([target.new_full((len(target), 1), BOS), target[:, :-1]], dim=1)


assert len({PAD, BOS, EOS}) == 3
assert {PAD, BOS, EOS}.isdisjoint(word_to_id.values())
assert decode_ids(encode_words(CAT_WORDS)) == list(CAT_WORDS)

The code prints the two answer-input lists before any training.
Later we will measure how one changed previous word affects the same writer, without changing its weights.


In [ ]:
first_guess = "cat"  # CHANGE THIS: a hypothetical wrong word; "mat" is correct.
correct_target = list(reversed(CAT_WORDS)) + ["<eos>"]
practice_inputs = ["<bos>"] + correct_target[:-1]
print("Canonical source:", CAT_SENTENCE)
print("Practice input -> answer:", list(zip(practice_inputs, correct_target)))
print("Hypothetical generation: <bos> -> guessed", first_guess, "-> next input", first_guess)
print("Correct previous input at step two:", practice_inputs[1])
print("Takeaway: a wrong guess changes the next input, not the stored source.")


**The catch: at generation time, there is no teacher to keep the inputs on track.**
Begin with `<bos>`, then feed each chosen word into the next step, even if it is wrong.
The model practised with correct history but must now use its own imperfect history.
That mismatch is **exposure bias**. Teacher forcing alone does not teach recovery from
every off-track history, so we test actual generation separately from practice loss.

## 3. Give both models the same controlled reading job

The cat-family vocabulary stays familiar, but generated source orders need not be grammatical.
This is an order/copy drill, not a corpus of English sentences.
Word IDs address trainable rows; they are not numerical word meanings.
Distinct `<pad>`, `<bos>` and `<eos>` IDs sit outside the ten word IDs.

Sources have two through eight words. We reserve forty unique sources at each length.
The remaining training budget is at most 1,200: length two has 100 possible orders, so only
60 can be used for training after reserving forty.
No source occurs in both splits. Both models receive the same batches in the same order.
This checks unseen orders at familiar lengths, not extrapolation to longer sources.

If we repeated only the cat sentence, the model could memorize one answer.
New lists test whether it can return the words of a different input in the right order.
The reversal function makes the answer key for practice and marking; it is not
a reversing shortcut in the reader or writer. During generation, neither model
is given the answer key.


<a id="corpus"></a>

## Worked details: Original split construction and frozen-corpus check


In [ ]:
rng=random.Random(31)
train,test={},{}
word_ids = tuple(word_to_id.values())
for length in range(2,9):
    unique=set()
    count=min(1200,len(word_ids)**length-40)
    while len(unique)<count+40:
        unique.add(tuple(rng.choice(word_ids) for _ in range(length)))
    rows=list(sorted(unique)); rng.shuffle(rows)
    # One length per bucket -> ID rectangles [N_train,L] and [40,L]; no blank slots are needed.
    train[length]=torch.tensor(rows[:count]); test[length]=torch.tensor(rows[count:])


In [ ]:
from encoder_decoder_example import load_cat_corpus

frozen_corpus = load_cat_corpus()
for split_name, generated_split in (("train", train), ("test", test)):
    for length, rows in generated_split.items():
        torch.testing.assert_close(rows, frozen_corpus[split_name][length], rtol=0, atol=0)
print("Original seeded source orders match the frozen main-demo corpus exactly.")


In [ ]:
for length in train:
    training_rows = {tuple(r) for r in train[length].tolist()}
    testing_rows = {tuple(r) for r in test[length].tolist()}
    assert training_rows.isdisjoint(testing_rows)
    assert len(testing_rows) == 40
    assert len(training_rows) == min(1200, len(word_ids)**length - 40)
    assert set(train[length].flatten().tolist() + test[length].flatten().tolist()) <= set(word_ids)
    print(f"Length {length}: {len(training_rows)} train, {len(testing_rows)} held out; "
          f"{len(word_ids)**length:,} possible word orders")
print("Takeaway: both models receive the same split; no test source appears in training.")
print("Generated word orders are a controlled access test, not a corpus of English sentences.")


**Your turn:** Change only `example_length` to inspect another length bucket before fitting.

In [ ]:
example_length = len(CAT_WORDS)  # CHANGE THIS: 2 through 8
example = test[example_length][0].tolist()
print("source words:", " ".join(decode_ids(example)))
print("required words:", " ".join(decode_ids(reversal_targets(torch.tensor([example]))[0])))


<a id="roles"></a>

## Worked details: Separate reader/writer roles and the full comparison model


## 4. Open the reader/writer before training it

**Predict:** when the writer produces its first word, which of these is its input:
the first source word, the last source word, or `<bos>` plus the reader's context?

Use `ReaderWriter(False)` first. In `step`, read the **no-look-back branch**.
It embeds the previous answer input, updates the writer's own state, and sends that state to the head.
`read` produces all source notes, but this first writer gets only the final one as its initial context.

The first version can keep useful order information, but the writer cannot ask
for a source detail again if the final note fails to preserve it. A source-access
branch is a possible repair for that restriction, not proof that every short
summary must fail. Its payoff is tested later on the same held-out task.

The same class also contains an optional source-access branch.
Leave that branch aside until after the forward, teacher-input and backward checks below.
We will then have a concrete reason to ask what it adds.

**Read the complete class in two passes.** The first pass follows only
`look_back=False` through source reading, previous inputs, loss and feedback.
Return to the extra branch at the [attention section](#attention), after asking
what source details the writer needs. Both branches stay defined in one place
for the controlled comparison; seeing the full code here does not make the
later mechanism a prerequisite for these first experiments.


In [ ]:
class ReaderWriter(nn.Module):
    def __init__(self,look_back):
        super().__init__(); self.look_back=look_back
        self.embedding=nn.Embedding(V,24)
        self.reader=nn.GRU(24,64,batch_first=True)
        self.writer=nn.GRUCell(24+(64 if look_back else 0),64)
        self.query=nn.Linear(64,64,bias=False) if look_back else None
        self.head=nn.Linear(128 if look_back else 64,V)
    def read(self,source):
        # Source IDs [B,L] -> rows [B,L,24] -> saved notes [B,L,64].
        # Final state [1,B,64] -> [B,64] drops the one-layer axis, not a source position.
        memory,last=self.reader(self.embedding(source)); return memory,last[0]
    def step(self,previous,state,memory):
        if self.look_back:
            # Project writer state [B,64] -> query [B,64]; unsqueeze -> [B,64,1].
            # Batched dot products [B,L,64] @ [B,64,1] -> [B,L,1] -> [B,L]; divide by sqrt(64) to tame score scale.
            scores=torch.bmm(memory,self.query(state).unsqueeze(2)).squeeze(2)/8
            # Normalize over source positions: [B,L] -> [B,L], one unit-sum reading budget per example.
            weights=scores.softmax(-1)
            # [B,L] -> [B,1,L] @ notes [B,L,64] -> [B,1,64] -> [B,64], a weighted source blend.
            context=torch.bmm(weights.unsqueeze(1),memory).squeeze(1)
            # Previous IDs [B] -> [B,24]; append context features [B,64] -> [B,88] for the next writer update.
            state=self.writer(torch.cat([self.embedding(previous),context],-1),state)
            # Join state/context features [B,64] + [B,64] -> [B,128]; project 128 -> V scores [B,V].
            logits=self.head(torch.cat([state,context],-1))
        else:
            # Without look-back, [B] -> [B,24] feeds the writer; its state [B,64] alone projects -> [B,V].
            state=self.writer(self.embedding(previous),state); logits=self.head(state); weights=None
        return logits,state,weights
    def forward(self,source,target_ids=None,*,teacher=True,detach_reader=False):
        if teacher:
            if target_ids is None:
                raise ValueError("Teacher forcing needs the correct answer IDs from the data.")
            if target_ids.ndim != 2 or len(target_ids) != len(source) or target_ids.shape[1] < 1:
                raise ValueError("Answers must be a nonempty [batch, writer time] tensor.")
            if target_ids.dtype != torch.long:
                raise TypeError("Answer IDs must use torch.long.")
            inputs = practice_prefix(target_ids)
            steps = target_ids.shape[1]
        else:
            if target_ids is not None:
                raise ValueError("Generation must not receive an answer tensor.")
            inputs = None
            steps = source.shape[1] + 1
        memory,state=self.read(source); previous=source.new_full((len(source),),BOS)
        if detach_reader:
            memory, state = memory.detach(), state.detach()
        scores=[]; maps=[]
        for t in range(steps):
            if teacher: previous=inputs[:,t]
            logits,state,weights=self.step(previous,state,memory); scores.append(logits)
            if weights is not None: maps.append(weights)
            # Generation selects guesses [B,V] -> [B]; practice only reads the shifted prefix.
            if not teacher: previous=logits.argmax(-1)
        # Stack adds writer time: T score arrays [B,V] -> [B,T,V]; reading maps [B,L] -> [B,T,L].
        return torch.stack(scores,1), torch.stack(maps,1) if maps else None


### 4.1 Follow one source read and the first writing step

**Watch the boundary, not just the names.** Six source words select six 24-number word rows.
The reader produces a 64-number note after each word. Its final 64 numbers initialize the writer.
The writer receives the `<bos>` row, makes a new 64-number writing state, and the head turns it
into thirteen word scores. None of those feature vectors is itself a vocabulary score.

The code samples only three coordinates to keep the values readable. Those coordinates have
no hand-assigned meanings such as 'cat' or 'last word'.
The parameter counts make the ownership visible: a shared word table, separate reader/writer
weights, and a separate head. Reusing a GRU *kind* is not reusing one GRU's weights.


In [ ]:
with torch.random.fork_rng(devices=[]):
    torch.manual_seed(31)
    mechanism_model = ReaderWriter(False)
walkthrough_source = torch.tensor([encode_words(CAT_WORDS)])
walkthrough_target = reversal_targets(walkthrough_source)
with torch.no_grad():
    source_rows = mechanism_model.embedding(walkthrough_source)
    reader_notes, reader_last = mechanism_model.reader(source_rows)
    writer_start = reader_last[0]
    start_ids = walkthrough_source.new_full((1,), BOS)
    first_scores, writer_after_start, _ = mechanism_model.step(start_ids, writer_start, reader_notes)
print("Source rows:", tuple(source_rows.shape), "| reader notes:", tuple(reader_notes.shape))
print("Final reader state = initial writer state:", tuple(writer_start.shape))
for word, note in zip(CAT_WORDS, reader_notes[0]):
    print(word, "-> reader feature sample", note[:3].tolist())
role_modules = {
    "word rows": mechanism_model.embedding, "encoder": mechanism_model.reader,
    "decoder": mechanism_model.writer, "head": mechanism_model.head,
}
role_parameters = {name: sum(p.numel() for p in module.parameters())
                   for name, module in role_modules.items()}
print("Different learned jobs:", role_parameters)
print("First writer input:", decode_ids(start_ids), "| scores:", tuple(first_scores.shape))
print("Untrained first guess:", decode_ids(first_scores.argmax(-1)),
      "| required first word:", decode_ids(walkthrough_target[:, 0]))


The reader's final note became the writer's *initial* note.
The first writer input was `<bos>`, not a copied source word. Its first guess is an initialized
model result, not musical or language skill. Same-width notes can carry information between
separately learned modules. The writer's next note is its own updated state, not a reread source.


<a id="previous-input"></a>

## Worked details: Change only the next input


### 4.2 What if that first guess were wrong?

Keep the first computed writer state and all weights fixed. For an inspectable intervention,
pretend the first guess were `cat`, even if the actual initialized model guessed something else.
Compare the second step with correct previous input `mat` versus hypothetical own input `cat`.
This changes one input, not the source, the first scores or an optimizer setting.


In [ ]:
with torch.no_grad():
    correct_previous = walkthrough_target[:, 0]
    wrong_previous = correct_previous.new_full((1,), word_to_id[first_guess])
    teacher_scores, teacher_state, _ = mechanism_model.step(
        correct_previous, writer_after_start, reader_notes,
    )
    own_guess_scores, own_guess_state, _ = mechanism_model.step(
        wrong_previous, writer_after_start, reader_notes,
    )
second_input_gap = (teacher_scores - own_guess_scores).abs().max().item()
print("Hypothetical first guess:", first_guess, "; the first scores and writer state remain unchanged.")
print("Next practice input:", decode_ids(correct_previous),
      "| next hypothetical generation input:", decode_ids(wrong_previous))
print("Largest next-score change:", second_input_gap)
assert second_input_gap == 0 if first_guess == "mat" else second_input_gap > 0
print("Takeaway: the input policy changes the next computation; it does not repair the earlier guess.")


The score difference belongs to the *next* computation.
The earlier answer is still graded against `mat`. Teacher forcing did not repair it.
**Your turn:** change only `first_guess` to `mat`, rerun its small setup and this check.
The gap becomes zero because the two next inputs now agree, not because the weights learned anything.


<a id="loss"></a>

## Worked details: Current-answer probabilities and loss


### 4.3 What tells these parts how to improve?

The training data gives a correct answer at each writing position.
Each prediction pays an error based on how little probability it assigned that answer.
A confident wrong choice pays more than a hesitant one. We combine these current-word errors;
we do not backpropagate through the spelling of an argmax guess.

Read the table below as **previous input -> current answer -> its error**.
Correct inputs can make later practice well-posed while the writer still makes wrong current guesses.


In [ ]:
with torch.no_grad():
    untrained_scores, _ = mechanism_model(
        walkthrough_source, target_ids=walkthrough_target, teacher=True,
    )
    answer_probabilities = untrained_scores.softmax(-1).gather(
        2, walkthrough_target.unsqueeze(-1),
    )[0, :, 0]
    answer_losses = F.cross_entropy(
        untrained_scores.flatten(0, 1), walkthrough_target.flatten(), reduction="none",
    )
for previous, answer, probability, error in zip(
    decode_ids(practice_prefix(walkthrough_target)[0]),
    decode_ids(walkthrough_target[0]), answer_probabilities.tolist(), answer_losses.tolist(),
):
    print(f"Input {previous:>5} | check {answer:>5} | probability {probability:.4f} | error {error:.4f}")
print("One example's mean error:", answer_losses.mean().item())
print("Takeaway: correct previous inputs do not excuse incorrect current predictions.")


<a id="gradients"></a>

## Worked details: Connected versus detached teaching routes


### 4.4 How can an encoder learn if it never gets a word grade?

There is no teacher-provided target vector for the summary. The target is the answer.
The decoder is the link between that answer error and the encoder's note: it tells us
which small note changes would help its current prediction. The encoder then receives
feedback about the weights that made that note, rather than a command to memorize a vector.

Its summary helped create the writer states and scores. Follow the error backward through
the head and writer to the initial writer state, then across that same handoff to the reader.
Now every source read that helped form the summary can contribute feedback to the reader's weights.
That is how a reader learns what information a later writer will need.

**Predict:** if we keep identical summary numbers but cut only their backward connection,
do the predictions change? Do the encoder's weight gradients change?
`detach` does exactly this controlled test: same values forward, no feedback through that route.

The shared word table also feeds the writer's previous tokens. Its gradient therefore need not
vanish when the reader route is detached. In this reversal task, source words are also answer inputs.
Do not use 'cat's shared row still has a gradient' as proof the encoder weights were taught.


In [ ]:
from copy import deepcopy


def gradient_norm(module):
    return sum(float(parameter.grad.square().sum())
               for parameter in module.parameters() if parameter.grad is not None) ** 0.5


training_probes, gradient_records, probe_scores, probe_losses = {}, {}, {}, {}
for detached in (False, True):
    trial = deepcopy(mechanism_model)
    scores, _ = trial(walkthrough_source, target_ids=walkthrough_target,
                      teacher=True, detach_reader=detached)
    loss = F.cross_entropy(scores.flatten(0, 1), walkthrough_target.flatten())
    loss.backward()
    modules = {"word rows": trial.embedding, "encoder": trial.reader,
               "decoder": trial.writer, "head": trial.head}
    gradient_records[detached] = {name: gradient_norm(module) for name, module in modules.items()}
    training_probes[detached], probe_scores[detached], probe_losses[detached] = (
        trial, scores.detach(), loss.item(),
    )
    print("Reader handoff detached:", detached, "| feedback norms:", gradient_records[detached])
torch.testing.assert_close(probe_scores[False], probe_scores[True], rtol=0, atol=0)
assert gradient_records[False]["encoder"] > 0 and gradient_records[True]["encoder"] == 0
assert gradient_records[True]["decoder"] > 0 and gradient_records[True]["word rows"] > 0
lookup_reader_feedback = (
    training_probes[False].embedding.weight.grad - training_probes[True].embedding.weight.grad
).norm().item()
print("Same forward scores/errors; changed backward route. Reader contribution to word-row feedback:",
      lookup_reader_feedback)
assert lookup_reader_feedback > 0
for name, parameter in mechanism_model.named_parameters():
    torch.testing.assert_close(parameter, training_probes[False].get_parameter(name), rtol=0, atol=0)


**Read the result in order:** the copied models give identical word scores and loss.
The connected copy has encoder feedback; the detached copy has none.
Decoder and head weights still receive feedback. The shared table still receives the writer's
lookup feedback; the difference between its two gradients isolates the extra reader route.
Those signed contributions can cancel, so a larger total norm is not the definition of a better route.

The code measured gradients, not learned weights. Every parameter is still equal to the
initialized model. Cutting a backward connection does not magically change the stored forward values.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
positions = np.arange(len(role_modules))
for detached, offset, label in ((False, -.18, "Connected handoff"), (True, .18, "Handoff detached")):
    ax.bar(positions + offset, list(gradient_records[detached].values()), width=.36, label=label)
ax.set_xticks(positions, list(role_modules))
ax.set(ylabel="Weight-gradient norm", title="Same prediction, but can its error teach the encoder?")
ax.legend(); fig.tight_layout()
fig.savefig(Path("images") / "03c-training-gradient-paths.png", dpi=160)
plt.show(); plt.close(fig)


<a id="optimizer"></a>

## Worked details: From gradients to changed weights


![The same initialized word predictions give encoder gradients only through a connected reader handoff; decoder/head gradients remain when it is detached.](images/03c-training-gradient-paths.png)

**Read the bars:** the encoder loses its teaching signal, not its forward summary values.
The shared word table still receives the writer's lookup feedback. This is a measured
single-example gradient comparison, not a trained-model quality score.

### 4.5 When do the weights actually change?

Backward fills each parameter's gradient: advice for a local weight change.
It does not itself rewrite the weights or edit a saved reader state.
Let Adam take one step on the connected copy, then read and write the same source again.
The new pass computes new states with updated reader, writer, word-table and head weights.
This single-example payoff verifies the connection, not generalization.


In [ ]:
learning_copy = training_probes[False]
before_step = {name: parameter.detach().clone() for name, parameter in learning_copy.named_parameters()}
one_step_optimizer = torch.optim.Adam(learning_copy.parameters(), lr=0.004)
one_step_optimizer.step()
with torch.no_grad():
    after_step_scores, _ = learning_copy(walkthrough_source, target_ids=walkthrough_target, teacher=True)
    after_step_loss = F.cross_entropy(after_step_scores.flatten(0, 1), walkthrough_target.flatten()).item()
update_sizes = {
    prefix: max((parameter.detach() - before_step[name]).abs().max().item()
                for name, parameter in learning_copy.named_parameters() if name.startswith(prefix + "."))
    for prefix in ("embedding", "reader", "writer", "head")
}
print("One copied example's error:", probe_losses[False], "->", after_step_loss)
print("Largest actual weight changes:", update_sizes)
assert after_step_loss < probe_losses[False] and all(change > 0 for change in update_sizes.values())
training_evidence = {
    "source_words": list(CAT_WORDS), "required_words": decode_ids(walkthrough_target[0]),
    "teacher_inputs": decode_ids(practice_prefix(walkthrough_target)[0]),
    "role_parameters": role_parameters, "source_rows_shape": list(source_rows.shape),
    "reader_notes_shape": list(reader_notes.shape), "writer_initial_shape": list(writer_start.shape),
    "next_input_score_gap": second_input_gap, "gradient_norms": gradient_records,
    "reader_lookup_feedback_norm": lookup_reader_feedback,
    "before_update_loss": probe_losses[False], "after_update_loss": after_step_loss,
    "weight_changes": update_sizes,
}
print("Takeaway: backward did not rewrite weights; one optimizer step trained all four learned jobs.")


<a id="encoder-only"></a>

## Worked details: Improve only the encoder


### Is the encoder learning a better note, or is only the decoder getting better?

The previous update changed both, so it could not separate those explanations.
Now freeze the word table, decoder and prediction head. Change **only the encoder weights**
on the mat/rug pair from section 1. Grade only the first required word for this small check.
Both decoder inputs are `<bos>`: teacher forcing has supplied no answer word yet.

First give both examples the same note deliberately; their first-word scores must match.
Then use their actual notes and take one encoder-only step. Check that the notes change
and the combined answer error falls while the decoder stays byte-for-byte unchanged.


In [ ]:
# %% 4.5 - Can a better message help an unchanged decoder?
encoding_copy = deepcopy(mechanism_model)
for parameter in encoding_copy.parameters():
    parameter.requires_grad_(False)
for parameter in encoding_copy.reader.parameters():
    parameter.requires_grad_(True)
contrast_words = [list(CAT_WORDS), [*CAT_WORDS[:-1], "rug"]]
contrast_sources = torch.tensor([encode_words(words) for words in contrast_words])
correct_first = contrast_sources[:, -1]
same_start = contrast_sources.new_full((2,), BOS)
frozen_before = {name: parameter.detach().clone()
                 for name, parameter in encoding_copy.named_parameters() if not parameter.requires_grad}


def first_word_outputs():
    memory, message = encoding_copy.read(contrast_sources)
    scores, _, _ = encoding_copy.step(same_start, message, memory)
    return message, scores


messages, first_scores = first_word_outputs()
with torch.no_grad():
    identical_scores, _, _ = encoding_copy.step(
        same_start, messages[:1].expand_as(messages), None,
    )
assert torch.equal(identical_scores[0], identical_scores[1])
initial_messages = messages.detach().clone()
initial_probabilities = first_scores.detach().softmax(-1).gather(1, correct_first[:, None])[:, 0]
encoding_loss = F.cross_entropy(first_scores, correct_first)
encoder_optimizer = torch.optim.SGD(encoding_copy.reader.parameters(), lr=0.1)
encoder_optimizer.zero_grad()
encoding_loss.backward()
encoder_optimizer.step()
with torch.no_grad():
    new_messages, new_scores = first_word_outputs()
    new_loss = F.cross_entropy(new_scores, correct_first).item()
    new_probabilities = new_scores.softmax(-1).gather(1, correct_first[:, None])[:, 0]
message_change = (new_messages - initial_messages).norm().item()
assert message_change > 0 and new_loss < encoding_loss.item()
assert all(torch.equal(parameter.detach(), frozen_before[name])
           for name, parameter in encoding_copy.named_parameters() if not parameter.requires_grad)
for words, before, after in zip(contrast_words, initial_probabilities, new_probabilities):
    print(" ".join(words), "| correct first word:", words[-1],
          f"| probability {before.item():.4f} -> {after.item():.4f}")
training_evidence["encoder_only"] = {
    "source_words": contrast_words, "first_answers": decode_ids(correct_first),
    "before_probabilities": initial_probabilities.tolist(), "after_probabilities": new_probabilities.tolist(),
    "before_loss": encoding_loss.item(), "after_loss": new_loss,
    "message_change_norm": message_change, "decoder_and_head_unchanged": True,
    "identical_message_score_gap": (identical_scores[0] - identical_scores[1]).abs().max().item(),
}
print("Only encoder weights changed; first-word error:", encoding_loss.item(), "->", new_loss)
print("Takeaway: better encoding means a message that helps this decoder's task, not a prescribed vector.")


**Follow the code:** the first part freezes everything except `reader`.
`first_word_outputs` recomputes the notes and lets the unchanged writer use them.
The identical-note check proves why the source distinction matters; the final checks
separate a changed note from changed decoder weights.

The improvement has a concrete meaning: these newly computed notes help this same decoder
assign more probability to the correct answers overall. No target encoding was supplied.
This checks one local learning step on two examples, not reliable reversal or generalization.
Full training changes both partners across many examples and checks held-out answers.


<a id="attention"></a>

## Worked details: Source attention and the two-model benchmark


### 4.6 That teaches a summary. Can the writer reopen a source detail?

The first writer still receives only one fixed-size summary.
Better training or LSTM retention can improve that summary; neither gives the writer direct
addresses for earlier source states. A short cat sentence may fit perfectly, so we must measure
the restriction rather than declare the one-card model doomed.

The reversal job stays unchanged: recover the input words, last to first.
Longer lists ask the same fixed-size note to preserve more details. Keeping the
task and examples the same lets us ask whether another way of reading the source
helps with those details, rather than changing the goal halfway through.

Keep the reader's note after each source word instead. At every writing step, project the current
writer state into a **query**, compare it with those notes, and blend them by the resulting weights.
Only now name the additional mechanism: **source attention**.
It changes where the writer may look, not the rule about which previous answer input it receives.


**Read this picture only after the basic handoff makes sense.**
Each source word has a saved contextual reader state. The writer's current state forms a
query; its match scores weight those saved states into a fresh context.
This is source access, not a claim that a literal spoken question exists inside a vector.

![The writer can query all six contextual reader states, then blend them for the current writing step.](images/03-evo-05-writer-looks-back.png)

The two occurrences of `the` have different addresses and different contextual notes.
The blend uses reader states, not raw IDs or one unchanging CBOW average.
Its weights sum to one over source positions; they are **not probabilities of the next answer word**.
The previous answer input and fresh context then update the writer, and the head scores the answer.


In [ ]:
models, histories, parameter_counts = {}, {}, {}
canonical_source = torch.tensor([encode_words(CAT_WORDS)])
canonical_target = reversal_targets(canonical_source)
assert decode_ids(canonical_target[0]) == ["mat", "the", "on", "sat", "cat", "the", "<eos>"]
assert practice_prefix(canonical_target)[0, 0].item() == BOS
assert torch.equal(practice_prefix(canonical_target)[:, 1:], canonical_target[:, :-1])
for look_back in [False, True]:
    torch.manual_seed(31)
    model = ReaderWriter(look_back)
    seen_inputs = []
    real_step = model.step

    def traced_step(previous, state, memory):
        seen_inputs.append(previous.detach().clone())
        return real_step(previous, state, memory)

    with torch.no_grad(), patch.object(model, "step", side_effect=traced_step):
        model(canonical_source, target_ids=canonical_target, teacher=True)
        assert torch.equal(torch.stack(seen_inputs, 1), practice_prefix(canonical_target))
        seen_inputs.clear()
        free_scores, _ = model(canonical_source, teacher=False)
        actual_inputs = torch.stack(seen_inputs, 1)
        assert bool((actual_inputs[:, 0] == BOS).all())
        assert torch.equal(actual_inputs[:, 1:], free_scores.argmax(-1)[:, :-1])
    optimizer = torch.optim.Adam(model.parameters(), lr=0.004)
    schedule = random.Random(99)
    batch_rng = torch.Generator().manual_seed(99)
    history = []
    for update in range(1000):
        length = schedule.randint(2, 8)
        rows = train[length]
        source = rows[torch.randint(len(rows), (64,), generator=batch_rng)]
        # Reverse [64,L] and append [64,1] end IDs -> [64,L+1], including the end prediction in training.
        target = reversal_targets(source)
        logits, _ = model(source, target_ids=target, teacher=True)
        # Scores [64,L+1,V] -> [64*(L+1),V], targets -> [64*(L+1)]; merge matching batch/time axes only.
        loss = F.cross_entropy(logits.flatten(0, 1), target.flatten())
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        history.append(loss.item())
    model.eval()
    models[look_back], histories[look_back] = model, history
    parameter_counts[look_back] = sum(p.numel() for p in model.parameters())
    print("look back:", look_back, "loss", round(history[0], 4), "->", round(history[-1], 4),
          "parameters", parameter_counts[look_back])


**Now practice many examples, without changing the comparison.**
The data builds each required reversal. `forward` receives those answer IDs explicitly for
teacher forcing; it does not infer the answer inside the model.
The loop grades raw word scores against the current targets, clears previous gradients,
runs backward, clips an oversized gradient, then lets Adam update weights.
Correct answer inputs do not remove any current-word errors from this loss.

Both models get 1,000 updates, batches of 64, the same schedules, and the same 24/64 feature widths.
Each independent source begins fresh reader/writer state; learned weights persist across examples.
The look-back model also has more parameters. A better score cannot isolate access from capacity or training luck.

**Your turn:** inspect the printed models. The optional query and wider writer input change source
access; they do not turn recurrence into a Transformer or teacher forcing into an architecture.


In [ ]:
look_back_to_inspect = False  # CHANGE THIS: True
print(models[look_back_to_inspect])


<a id="source-memory-check"></a>

### Three writing steps, the same source notes

**Predict:** after a source note helps one answer step, is it altered or unavailable
to the next step?

Use a copy of the just-trained look-back model. Read the cat source once, then supply
the first three correct previous-answer inputs. Keeping that input history fixed lets
us inspect the memory access without also comparing generation mistakes.
This is a read-only mechanism check, not another training or answer-quality run.

The table reports the largest changed entry in the source notes, writing state and
attention weights. The first attention change has no earlier reading to compare with,
so it is shown as `--`, not zero.

In [ ]:
from copy import deepcopy

memory_probe_model = deepcopy(models[True]).eval()
memory_probe_source = torch.tensor([encode_words(CAT_WORDS)])
memory_probe_parameters = {
    name: parameter.detach().clone()
    for name, parameter in memory_probe_model.named_parameters()
}
memory_probe_rows, memory_probe_last_weights = [], None
with torch.no_grad():
    memory_probe_notes, memory_probe_state = memory_probe_model.read(memory_probe_source)
    memory_probe_original = memory_probe_notes.clone()
    memory_probe_previous = practice_prefix(reversal_targets(memory_probe_source))[:, :3]
    for memory_probe_index in range(memory_probe_previous.shape[1]):
        memory_probe_input = memory_probe_previous[:, memory_probe_index]
        memory_probe_old_state = memory_probe_state.clone()
        memory_probe_scores, memory_probe_state, memory_probe_weights = memory_probe_model.step(
            memory_probe_input, memory_probe_state, memory_probe_notes,
        )
        if memory_probe_weights is None:
            raise RuntimeError("The look-back model returned no source-attention reading.")
        memory_probe_rows.append({
            "step": memory_probe_index + 1,
            "previous": decode_ids(memory_probe_input)[0],
            "chosen": decode_ids(memory_probe_scores.argmax(-1))[0],
            "source_change": (memory_probe_notes - memory_probe_original).abs().max().item(),
            "writer_change": (memory_probe_state - memory_probe_old_state).abs().max().item(),
            "attention_change": None if memory_probe_last_weights is None else (
                memory_probe_weights - memory_probe_last_weights
            ).abs().max().item(),
        })
        memory_probe_last_weights = memory_probe_weights.clone()
assert torch.equal(memory_probe_notes, memory_probe_original)
assert all(torch.equal(parameter, memory_probe_parameters[name])
           for name, parameter in memory_probe_model.named_parameters())
print("step | previous | chosen | source change | writer change | attention change")
for memory_probe_row in memory_probe_rows:
    memory_probe_change = "--" if memory_probe_row["attention_change"] is None else f'{memory_probe_row["attention_change"]:.6f}'
    print(f'{memory_probe_row["step"]:>4} | {memory_probe_row["previous"]:>8} | {memory_probe_row["chosen"]:>6} | '
          f'{memory_probe_row["source_change"]:.6f} | {memory_probe_row["writer_change"]:.6f} | {memory_probe_change}')

**Read the code in three groups.** First copy the trained model and read once.
Then run three existing `step` calls while retaining the returned writing state.
Finally check exact source equality and unchanged parameter values.

The source-change column stays zero; the assertions check the actual tensors, not just
rounded display values. The writing state and attention reading can change while those
same notes remain available. No source entry is crossed off.
The original trained models and their recorded comparisons are untouched.

This is not a rule that every query must differ or every source position must be used once.
The source notes would be recomputed for a new input; training can also change the weights
that produce them.

<a id="evaluation"></a>

## Worked details: Held-out generation versus teacher-forced checking


## 5. Can it write without the teacher's history?

The training loss asked about every current word under correct previous-answer inputs.
Now also run generation: no answer tensor, only `<bos>` and the model's own earlier guesses.
Weights are fixed during checking; this is not another optimizer pass.

**This is the point of saving new test lists:** can the writer return each one
backwards without being told the answer words? A right answer must depend on
that input, not just repeat the familiar cat response. Success here supports
this small reading-and-reordering skill at the tested lengths, not general
language understanding or a guarantee for much longer lists.

Check two units separately:
- **Whole-answer accuracy:** every word and `<eos>` must match. One wrong word fails the row.
- **Token accuracy:** count the individual correct words, so partly wrong answers still tell us something.

Equal whole-answer scores do not imply equal histories. Once the first mistake fails a row,
that unit cannot reveal how many additional mistakes the wrong history caused.
The token scores below can.
The known source length supplies a bounded generation budget including end, not the answer words.


In [ ]:
results = []
with torch.no_grad():
    for look_back, model in models.items():
        for length, source in test.items():
            # The held-out answers use the same layout: reverse [B,L], append end [B,1] -> [B,L+1].
            target = reversal_targets(source)
            tf_scores, _ = model(source, target_ids=target, teacher=True)
            free_scores, _ = model(source, teacher=False)
            # Choose IDs [B,L+1,V] -> [B,L+1]; all(time) -> [B] whole-answer checks, mean -> scalar accuracy.
            # The token scores below instead average every [B,L+1] check, so partial answers still count there.
            tf_ok = (tf_scores.argmax(-1) == target).all(1).float().mean().item()
            free_ok = (free_scores.argmax(-1) == target).all(1).float().mean().item()
            tf_token = (tf_scores.argmax(-1) == target).float().mean().item()
            free_token = (free_scores.argmax(-1) == target).float().mean().item()
            results.append((look_back, length, tf_ok, free_ok, tf_token, free_token))
print("look-back length practice-sequence free-sequence practice-token free-token")
for row in results:
    print(row[0], row[1], *[f"{value:.2%}" for value in row[2:]])
# Seven per-length scores -> one equally weighted mean; this is not a pooled token accuracy.
fixed_mean = np.mean([r[3] for r in results if not r[0]])
attention_mean = np.mean([r[3] for r in results if r[0]])
print(f"Mean over lengths, free-running: one card {fixed_mean:.1%}; look-back {attention_mean:.1%}")
aggregate_scores = {}
for mode in models:
    rows = [row for row in results if row[0] == mode]
    token_counts = [(row[1] + 1) * len(test[row[1]]) for row in rows]
    aggregate_scores[mode] = {
        "practice_sequence": float(np.mean([row[2] for row in rows])),
        "free_sequence": float(np.mean([row[3] for row in rows])),
        "practice_token_pooled": float(np.average([row[4] for row in rows], weights=token_counts)),
        "free_token_pooled": float(np.average([row[5] for row in rows], weights=token_counts)),
    }
    print("Aggregate look-back", mode, {key: f"{value:.2%}" for key, value in aggregate_scores[mode].items()})
if attention_mean > fixed_mean:
    print("Takeaway: look-back improved this held-out run; this is not a universal ranking.")
else:
    print("Takeaway: look-back did not improve this run; access exists but payoff was not established.")
fig, ax = plt.subplots(figsize=(6, 3))
for mode, label, color in [(False, "One learned card", "#ef8a23"), (True, "Writer looks back", "#2c7fb8")]:
    rows = [r for r in results if r[0] == mode]
    ax.plot([r[1] for r in rows], [r[3] for r in rows], "o-", label=label, color=color)
ax.set(xlabel="Source length", ylabel="Complete free-running accuracy", ylim=(0, 1.05))
ax.set_title("Word-order reversal: cat-family controlled access test")
ax.legend(); fig.tight_layout()
fig.savefig(Path("images") / "03c-cat-accuracy-by-length.png", dpi=160)
plt.show(); plt.close(fig)


The look-back model also has more weights, so this isn't attention alone.

**Read the two scores separately:** test each length group twice, without changing weights.
Whole-answer accuracy asks whether every token was right. Token accuracy shows what happened
inside answers that were partly wrong. Two runs can tie on whole answers yet make different
later mistakes. The plot shows only measured generation scores, without the answer key.

![Actual complete free-running accuracy for the cat-family word-order task, by source length.](images/03c-cat-accuracy-by-length.png)
*Saved counterpart of the measured plot above; the two models also differ in capacity.*

**Your turn:** Change only the first word in `source_to_reverse` to another `CAT_FAMILY_WORDS`
word. The default source is `the cat sat on the mat`; the required answer is
`mat the on sat cat the <eos>`. Generate without seeing the answer.
This probe is not automatically independent of training data; its split membership is printed.

In [ ]:
source_to_reverse = list(CAT_WORDS)  # CHANGE THIS: first word only, from CAT_FAMILY_WORDS.
source = torch.tensor([encode_words(source_to_reverse)])
print("Source words:", " ".join(source_to_reverse))
probe_answers = {}
with torch.no_grad():
    for mode, model in models.items():
        scores, _ = model(source, teacher=False)
        probe_answers[mode] = decode_ids(scores.argmax(-1)[0])
        print("look back", mode, "generated words:", " ".join(probe_answers[mode]))
print("Required words:", " ".join(decode_ids(reversal_targets(source)[0])))
source_key = tuple(source[0].tolist())
print("Probe membership: training", source_key in {tuple(row) for row in train[len(source_to_reverse)].tolist()},
      "; held out", source_key in {tuple(row) for row in test[len(source_to_reverse)].tolist()})


<a id="routing"></a>

## Worked details: Measured source-attention routing


**We know whether the answer was right. Where did the writer look?**

## 6. Inspect the route without declaring victory from a picture

**Predict:** For `the cat sat on the mat`, should reversal read source positions from last to first?
Two `the` tokens make content alone an ambiguous address: `the @ 0` differs from `the @ 4`.
The required source addresses are `5, 4, 3, 2, 1, 0`; `<eos>` has no single required source address. Recurrent reader states contain ordered
context. The learned map need not be a perfect anti-diagonal, a line running from upper right to
lower left. This heatmap comes from real free-running query weights, not hand-drawn target routing.

In [ ]:
with torch.no_grad():
    probe = source
    logits, weights = models[True](probe, teacher=False)
    generated = logits.argmax(-1)[0].tolist()
    # Choose one example: [1,L+1,L] -> [L+1,L], writer steps as rows and source positions as columns.
    heat = weights[0].numpy()
source_labels = [f"{word} @ {position}" for position, word in enumerate(source_to_reverse)]
generated_words = decode_ids(generated)
expected_addresses = list(reversed(range(probe.shape[1])))
assert heat.shape == (len(generated_words), len(source_labels))
assert np.allclose(heat.sum(1), 1.0)
fig, ax = plt.subplots(figsize=(9, 4.5))
image = ax.imshow(heat, vmin=0, vmax=1, cmap="Blues", aspect="auto")
ax.set_xticks(range(len(source_labels)), source_labels)
ax.set_yticks(range(len(generated_words)), [f"step {i}: chose {word}" for i, word in enumerate(generated_words)])
ax.set(xlabel="Reader source word and position", ylabel="Writer step",
       title="Measured free-running weights: diagnostic, not proof")
fig.suptitle("Source: " + " ".join(source_to_reverse))
fig.colorbar(image, ax=ax, label="Read weight"); fig.tight_layout()
fig.savefig(Path("images") / "03c-cat-attention-map.png", dpi=160)
plt.show(); plt.close(fig)
# Sum only source columns [L+1,L] -> [L+1], one reading budget at each writer step.
print("Row sums:", heat.sum(1).round(6).tolist())
print("Most-read source positions:", heat.argmax(1).tolist())
print("Required word addresses:", [source_labels[address] for address in expected_addresses],
      "; <eos> has no single source address")
print("Most-read words and addresses:", [source_labels[address] for address in heat.argmax(1)])
print("Takeaway: inspect routing beside generated answers; the picture alone cannot prove correctness.")


![Actual free-running cat-probe attention weights with generated words and distinct source addresses.](images/03c-cat-attention-map.png)
*Saved measured diagnostic, not ideal target routing. A correct answer need not have perfect anti-diagonal attention.*

**Your turn:** Change only `writing_step`, from zero through the actual final decoder step.
Does its most-read word and source address match the required reversal address?
The last required answer is `<eos>`, which has no single source address.

In [ ]:
writing_step = 0  # CHANGE THIS: any step from 0 through len(generated_words) - 1.
assert 0 <= writing_step < len(generated_words)
expected_address = source_labels[expected_addresses[writing_step]] if writing_step < len(expected_addresses) else "<eos>: no single source address"
print("Chose:", generated_words[writing_step], "; most-read:", source_labels[int(heat[writing_step].argmax())],
      "; required address:", expected_address)


## 7. What can you now explain and prove?

The goal was not to build a complicated substitute for Python's list reversal.
It was to make the reader/writer handoff easy to test, then hold the job fixed
while checking practice, unseen answers, and access to source notes.

| Question | Evidence |
|---|---|
| How do encoder and decoder differ? | Same update family, separate learned jobs and weight sets |
| What crosses their handoff? | Final reader state initializes writer state; word scores come later |
| What does teacher forcing change? | Correct previous input; the controlled next-input check changes later scores |
| How does the reader learn? | Writer loss reaches reader weights only while the backward handoff is connected |
| Does backward update weights? | No; the copied-model optimizer step changes them and lowers this example's loss |
| What does attention change? | A new weighted read of all saved source states at each writing step |
| Does that guarantee the answer? | No; grade actual held-out generation and inspect maps beside it |

| Teaching mechanism | Larger-system counterpart | What changes | What stays |
|---|---|---|---|
| Final reader state starts writer | Recurrent fixed-summary model | State width and data | Learned context crosses a handoff |
| Writer queries source notes | Recurrent source attention | Score rule and source masks | Query-dependent source access |
| Shifted correct answer inputs | Teacher-forced autoregressive training | Batch/target lengths | Current answer is not its own input |
| Separate reader/writer jobs | Transformer encoder-decoder | Attention replaces recurrent updates | Source/answer roles and supervised alignment |


In [ ]:
evidence = {"fixed_summary_mean_free_sequence_accuracy": round(float(fixed_mean), 6),
       "attention_mean_free_sequence_accuracy": round(float(attention_mean), 6),
       "held_out_sequences_per_length": len(test[2]), "source_lengths": list(test),
       "word_vocabulary_size": len(word_to_id), "total_vocabulary_with_controls": V,
       "parameters": parameter_counts, "aggregate_scores": aggregate_scores,
       "per_length_scores": results, "probe_generated_words": probe_answers,
       "source_words": source_to_reverse, "source_addresses": source_labels,
       "required_words": decode_ids(reversal_targets(probe)[0]),
       "most_read_addresses": heat.argmax(1).tolist(), "required_addresses": expected_addresses,
       "training_walkthrough": training_evidence}
Path("images", "03c-cat-measured-evidence.json").write_text(
    json.dumps(evidence, indent=2) + "\n", encoding="utf-8",
)
print(evidence)
print(f"Elapsed CPU wall time: {time.perf_counter() - started:.1f}s")


### What we built, explained and left for later

- **Built and measured:** reader/writer state and parameter ownership; shifted answer inputs;
  current-word loss; connected versus detached reader gradients; one copied-model optimizer step;
  independent source splits; summary/source-attention training; generation and routing diagnostics.
- **Explained:** why correct practice history differs from own-guess history; how shared word rows
  receive both reader and writer feedback; why source access differs from retention.
- **Deferred:** gate internals already taught in memory, scheduled sampling, beam search, mixed-length
  masking, source and loss masks, why complete-source encoders may use both directions
  while generated history has no future answers, longer-length extrapolation,
  multi-seed rankings and production translation.

Those masking and visibility topics are forward references, not explanations or
implementations hidden in this lab. Its source batches have equal lengths and its
reader is one-way. [Transformer Part 4A](../../04-transformers/04a-decoder-only-language-model-theory.ipynb)
owns target visibility and mask roles; [Part 5A](../../04-transformers/05a-encoder-decoder-and-cross-attention-theory.ipynb)
owns separate source memory and its visibility rules.

Do not confuse a state handoff with shared weights, correct previous inputs with corrected current
guesses, a detached forward value with a teachable reader, or a bright attention square with proof.


<a id="adapter"></a>

## Worked details: Use the inherited training loop


## 8. Keep the reading/writing job inside the model, not the trainer

Our common trainer needs word scores and aligned answers.
`ReaderWriter` also returns attention maps, so a small registered wrapper leaves those maps out.
It now **passes the supplied answer IDs through** for teacher forcing.
The dataset creates the reversal labels; the model does not manufacture an answer from the source.

Targets already contain each required word followed by `<eos>`.
They do not contain an extra `<bos>` label to remove, so the inherited alignment method stays unchanged.
The reader/writer itself shifts those correct targets into previous-answer inputs.

```text
data: source + correct answer IDs
shared loop -> wrapper -> reader/writer scores
current-word loss -> backward -> word table + reader + writer + head
optimizer -> changed weights for the next forward pass
```

Test this on a deep copy. The original comparison models keep every parameter unchanged.
The example data and generation budget are still a reversal drill, not a production translation system.


In [ ]:
from copy import deepcopy
from language_model import TrainableLanguageModel
from torch.utils.data import DataLoader, TensorDataset


class ReaderWriterTrainingModel(TrainableLanguageModel):
    def __init__(self, reader_writer):
        super().__init__()
        self.reader_writer = reader_writer

    def forward(self, source_ids, target_ids=None):
        # Keep scores [B,L+1,V] for the shared loss; reading maps are a diagnostic, not a second prediction target.
        logits, _ = self.reader_writer(source_ids, target_ids=target_ids, teacher=True)
        return logits


adapter = ReaderWriterTrainingModel(deepcopy(models[False]))
adapter_source = train[len(CAT_WORDS)][:8]
# Reverse [8,L] and append end IDs [8,1] -> [8,L+1], aligned with the writer's output steps.
adapter_target = reversal_targets(adapter_source)
adapter_loader = DataLoader(TensorDataset(adapter_source, adapter_target), batch_size=8)
adapter_optimizer = torch.optim.Adam(adapter.parameters(), lr=0.001)
original_embedding = models[False].embedding.weight.detach().clone()
original_parameters = {name: parameter.detach().clone()
                       for name, parameter in models[False].named_parameters()}
adapter_before = adapter.reader_writer.embedding.weight.detach().clone()

assert ReaderWriterTrainingModel.fit_language_model is TrainableLanguageModel.fit_language_model
adapter_losses = adapter.fit_language_model(
    adapter_loader, nn.CrossEntropyLoss(), adapter_optimizer, epochs=1,
)
assert torch.equal(models[False].embedding.weight.detach(), original_embedding)
assert all(torch.equal(parameter.detach(), original_parameters[name])
           for name, parameter in models[False].named_parameters())
adapter_change = (adapter.reader_writer.embedding.weight.detach() - adapter_before).abs().max().item()
assert adapter_change > 0
print("Inherited encoder-decoder training loss:", adapter_losses[0])
print("Embedding updated through reader and writer; original experiment unchanged:", adapter_change)


**Follow the call:** the common loop passes sources and answers to the wrapper.
The wrapper passes the answers to the same reader/writer whose input rule we now understand.
Only attention maps are left out of the loss. Feedback still reaches all registered learned parts.
The final equality check covers every original parameter, not just the word table.

You can now trace both directions: source words into reader notes, context into the writer,
previous-answer inputs into word scores, and current-answer errors back into both components.
The optimizer changes reusable weights; reading a new independent source computes fresh states.

We added source access without removing sequential recurrent reading and writing.
[Stage 04](../../04-transformers/README.md) keeps the roles and alignment, but uses attention,
positions and masks inside the complete Transformer blocks.
